# PCS Data Migration
There are two tables "marpower__150000propulsion__pcs_aft" and "marpower__150000propulsion__pcs_fwd". For a while, these got ingested by a python application that skipped mapping of PascalCase to snake_case. Before enabling the new data ingestion the old (PascalCase) tables are backed up with _old suffixes.
In this notebook this data is migrated: 
- marpower__150000propulsion__pcs_aft_old to marpower__150000propulsion__pcs_aft
- marpower__150000propulsion__pcs_fwd_old to marpower__150000propulsion__pcs_fwd

In [1]:
import polars as pl
import re

In [2]:
def format_col_name(name: str) -> str:
    # Replace dot with double underscore
    name = name.replace(".", "__")
    
    # 1. Insert underscore between lower-Upper (e.g., 'Above' and 'Is')
    name = re.sub(r'([a-z])([A-Z])', r'\1_\2', name)
    
    # 2. Handle abbreviations: Upper sequence followed by Upper+lower (e.g., 'POD' and 'Control')
    name = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', name)
    
    # 3. Insert underscore between letter-number and number-letter
    name = re.sub(r'([a-zA-Z])([0-9])', r'\1_\2', name)
    name = re.sub(r'([0-9])([a-zA-Z])', r'\1_\2', name)
    
    # Convert to lowercase
    formatted = name.lower()
    
    return formatted.replace("time_stamp", "timestamp")

In [5]:
uri_from = "postgresql://greptime-zero.tail0b4840.ts.net:4003/public"
#uri_to =  "postgresql://localhost:4003/public"
uri_to =  uri_from

tables = {
    t:pl.read_database_uri(query=f'SELECT * FROM public."{t}"', uri=uri_from, engine="adbc")
    for t in ["marpower__150000propulsion__pcs_aft_old","marpower__150000propulsion__pcs_fwd_old"]
}

In [8]:
for table, df_input in tables.items():
    # Use latest timestamp in fields
    # Map columns from PascalCase to snake_case
    # Drop old timestamp column
    timestamp_columns = [c for c in df_input.columns if c.endswith("TimeStamp")]
    df_input.with_columns(pl.max_horizontal(timestamp_columns).alias("timestamp"))\
    .rename(format_col_name)\
    .drop("greptime_timestamp") \
    .write_database(
        table_name=table.replace("_old",""),
        connection=uri_to,
        engine="sqlalchemy",
        if_table_exists="append"
    )  

- Get latest timestamp from data
- Rename columns

In [6]:
print(tables["marpower__150000propulsion__pcs_aft_old"])
print(tables["marpower__150000propulsion__pcs_fwd_old"])

shape: (50_683, 318)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ greptime_ ┆ ActionLev ┆ ActionLev ┆ ActionLev ┆ … ┆ TotalSoCA ┆ TotalSoCA ┆ TotalSoCA ┆ topic    │
│ timestamp ┆ el1tm12Ac ┆ el1tm12Ac ┆ el1tm12Ac ┆   ┆ vailBatte ┆ vailBatte ┆ vailBatte ┆ ---      │
│ ---       ┆ tive.HasV ┆ tive.IsVa ┆ tive.Time ┆   ┆ ries.IsVa ┆ ries.Time ┆ ries.Valu ┆ str      │
│ datetime[ ┆ alu…      ┆ lid       ┆ Sta…      ┆   ┆ lid       ┆ Sta…      ┆ e         ┆          │
│ μs]       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆          │
│           ┆ bool      ┆ bool      ┆ str       ┆   ┆ bool      ┆ str       ┆ i64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2026-06-0 ┆ true      ┆ true      ┆ 2026-06-0 ┆ … ┆ true      ┆ 2026-06-0 ┆ 53        ┆ marpower │
│ 4 11:03:4 ┆           ┆           ┆ 4T11:04:2 ┆   ┆           ┆ 4T11